# COLMAP

## Sparse Reconstruction

In [3]:
import pathlib
import pycolmap

output_path= pathlib.Path("../results/colmap_miniature")
image_dir= pathlib.Path("../data/images/miniature/")

output_path.mkdir()
database_path = output_path / "database.db"

pycolmap.extract_features(database_path, image_dir)
pycolmap.match_exhaustive(database_path)
maps = pycolmap.incremental_mapping(database_path, image_dir, output_path)
maps[0].write(output_path)

## Visualizing 

In [1]:
import sys
sys.path.append("../utils")
from colmap import load_colmap_model, create_open3d_geometries_from_colmap, save_colmap_geometries_ply
import open3d as o3d
import numpy as np

MODEL_DIR = "../results/colmap_miniature/0"   # change to "1" for the second model
PLY_OUT   = "../results/colmap_miniature/reconstruction.ply"

# Load model and base point cloud
cams, images, points = load_colmap_model(MODEL_DIR)
geometries = create_open3d_geometries_from_colmap(MODEL_DIR, scale_frustum=0.05)
pcd = geometries[0] if geometries else o3d.geometry.PointCloud()

# Create camera center spheres instead of line frustums
spheres = []
sphere_radius = 0.01
for img_id in sorted(images.keys()):
    info = images[img_id]
    qvec = info["qvec"]  # qw, qx, qy, qz
    tvec = info["tvec"]
    qw, qx, qy, qz = qvec
    q = np.array([qw, qx, qy, qz], dtype=float)
    w, x, y, z = q
    R = np.array([
        [1 - 2 * (y * y + z * z),     2 * (x * y - z * w),     2 * (x * z + y * w)],
        [    2 * (x * y + z * w), 1 - 2 * (x * x + z * z),     2 * (y * z - x * w)],
        [    2 * (x * z - y * w),     2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ], dtype=float)
    C = (-R.T @ tvec).ravel()
    sph = o3d.geometry.TriangleMesh.create_sphere(radius=sphere_radius)
    sph.translate(C)
    sph.compute_vertex_normals()
    sph.paint_uniform_color([0.1, 0.6, 0.9])
    spheres.append(sph)

print(f"Point cloud: {len(pcd.points)} points; Camera spheres: {len(spheres)}")

# Save points + camera spheres
out = save_colmap_geometries_ply([pcd] + spheres, PLY_OUT)
print(f"Saved PLY to: {out}")

# Visualize
o3d.visualization.draw_geometries(
    [pcd] + spheres,
    window_name="COLMAP Points and Camera Centers",
    width=1280,
    height=720,
)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Point cloud: 20838 points; Camera spheres: 103
Saved PLY to: ..\results\colmap_miniature\reconstruction.ply
